# Transaction data transformation

This notebook transforms the cleaned transaction data into a dataset containing useful analytical features

## Objectives

- Create temporal features from the transaction timestamp
- Represent purchases and refunds with an appropriate sign
- Exclude declined transactions from executed financial amounts
- Practise vectorised pandas operations
- Prepare the transaction data for later table joins and customer analysis

In [1]:
from pathlib import Path 
import pandas as pd

In [2]:
current_directory = Path.cwd()

if current_directory.name == "notebooks":
    project_root = current_directory.parent
else:
    project_root = current_directory

project_root

WindowsPath('c:/Users/Focus/Documents/GitHub/transaction-intelligence')

In [3]:
clean_transactions_path = (  project_root / "data" / "processed" / "transactions_clean.csv")
clean_transactions_path

WindowsPath('c:/Users/Focus/Documents/GitHub/transaction-intelligence/data/processed/transactions_clean.csv')

In [4]:
clean_transactions_path.exists()

True

## Loading the cleaned transactions

The cleaned transaction dataset will be used as the starting point for feature creation

The timestamp column is parsed directly as a pandas datetime column

In [6]:
transactions = pd.read_csv(clean_transactions_path, parse_dates = ["timestamp"])

transactions.head()

,transaction_id,account_id,merchant_id,timestamp,amount,currency,transaction_type,status
0,T0001,A001,M001,2025-03-01 08:15:00,34.80,EUR,Purchase,Completed
1,T0002,A003,M002,2025-04-01 10:42:00,4.20,EUR,Purchase,Completed
2,T0003,A005,M003,2025-05-01 19:10:00,18.99,EUR,Purchase,Completed
3,T0004,A006,M004,2025-06-01 21:35:00,249.90,EUR,Purchase,Completed
4,T0005,A008,M005,2025-08-01 12:05:00,62.40,EUR,Purchase,Completed


In [7]:
transactions.dtypes

transaction_id                 str
account_id                     str
merchant_id                    str
timestamp           datetime64[us]
amount                     float64
currency                       str
transaction_type               str
status                         str
dtype: object

In [10]:
transactions.shape

(52, 8)

In [11]:
transactions.info()

<class 'pandas.DataFrame'>
RangeIndex: 52 entries, 0 to 51
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   transaction_id    52 non-null     str           
 1   account_id        52 non-null     str           
 2   merchant_id       52 non-null     str           
 3   timestamp         52 non-null     datetime64[us]
 4   amount            52 non-null     float64       
 5   currency          52 non-null     str           
 6   transaction_type  52 non-null     str           
 7   status            52 non-null     str           
dtypes: datetime64[us](1), float64(1), str(6)
memory usage: 3.4 KB


In [12]:
transactions.isna().sum()

transaction_id      0
account_id          0
merchant_id         0
timestamp           0
amount              0
currency            0
transaction_type    0
status              0
dtype: int64

In [13]:
# Copy created for the transformations

transactions_features = transactions.copy(deep = True)

In [14]:
transactions_features.shape

(52, 8)

## Temporal features

The timestamp contains several pieces of information. I extract them into separate columns so that they can be analysed more easily

In [ ]:
transactions_features = transactions_features.assign(
    year=transactions_features["timestamp"].dt.year,
    month=transactions_features["timestamp"].dt.month,
    month_name=(
        transactions_features["timestamp"]
        .dt.month_name()
    ),
    weekday=(
        transactions_features["timestamp"]
        .dt.day_name()
    ),
    hour=transactions_features["timestamp"].dt.hour,
)

In [16]:
transactions_features[
    [
        "transaction_id",
        "timestamp",
        "year",
        "month",
        "month_name",
        "weekday",
        "hour",
    ]
].head()

,transaction_id,timestamp,year,month,month_name,weekday,hour
0,T0001,2025-03-01 08:15:00,2025,3,March,Saturday,8
1,T0002,2025-04-01 10:42:00,2025,4,April,Tuesday,10
2,T0003,2025-05-01 19:10:00,2025,5,May,Thursday,19
3,T0004,2025-06-01 21:35:00,2025,6,June,Sunday,21
4,T0005,2025-08-01 12:05:00,2025,8,August,Friday,12


## Financial features

The original amount is always positive; I create different representations depending on the financial meaning of the transaction

In [17]:
transactions_features["absolute_amount"] = (transactions_features["amount"].abs())

In [20]:
transactions_features[["transaction_id","amount","absolute_amount",]].head()

,transaction_id,amount,absolute_amount
0,T0001,34.80,34.80
1,T0002,4.20,4.20
2,T0003,18.99,18.99
3,T0004,249.90,249.90
4,T0005,62.40,62.40


In [21]:
# A purchase represents an outflow of money and a return represents the opposite movement
transaction_sign = {
    "Purchase": 1,
    "Refund": -1
}

In [24]:
transactions_features["transaction_sign"] = (transactions_features["transaction_type"].map(transaction_sign))

In [25]:
transactions_features[
    [
        "transaction_id",
        "transaction_type",
        "transaction_sign",
    ]
].tail()

,transaction_id,transaction_type,transaction_sign
47,T0056,Purchase,1
48,T0057,Purchase,1
49,T0058,Purchase,1
50,T0059,Purchase,1
51,T0060,Refund,-1


In [26]:
transactions_features[
    "transaction_sign"
].isna().sum()

np.int64(0)

In [27]:
# NOS HEMOS QUEDADO EN EL 12. Crear el importe con signo